# Pandas: Daten bereinigen, zusammenführen und auswerten
### Unterrichtsmaterial | Julia Brutskaya

---
**Kontext:** Kurs-Umfrage der Datenwald-Akademie, TechAcademy GmbH und der Akademie-Laden<br>
**Bibliotheken:** `pandas`, `numpy`, `matplotlib`

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="130"> <img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats Maulwurf" width="130"> <img src="png%20Dateien/der_ausreisser.png" alt="Der Ausreißer" width="130">

Neulich ging es um Datenqualität, Kennzahlen, Pivot-Tabellen und den One-Pager. Jetzt machst du dasselbe mit Pandas: erst die Daten bereinigen, dann Tabellen zusammenführen, dann auswerten. Am Ende steht ein eigenes Projekt mit Bericht für die Geschäftsführung.

| Block | Inhalt |
|---|---|
| A | Lücken und Dubletten |
| B | Datentypen, unplausible Werte, Schreibvarianten |
| C | Tabellen zusammenführen mit `merge` |
| D | Die Pivot-Tabelle |
| E | Das Simpson-Paradoxon |
| F | Aufwärmübungen und das Projekt „Akademie-Laden" |

---
# Block A – Lücken und Dubletten

## 1 – Die Kurs-Umfrage laden

Dieselbe Umfrage wie seit Tag 5. Heute bauen wir absichtlich Fehler hinein, um zu sehen, was sie mit den Kennzahlen machen.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

zeilen = [
    # teilnehmer_id, kurszeit, getraenk, tshirt, haustiere, anfahrt_min, lernstunden, punkte
    (101, "Vormittag", "Kaffee", "M", 0, 25, 3, 50), (102, "Abend", "Tee", "S", 1, 35, 4, 58),
    (103, "Vormittag", "Mate", "L", 0, 18, 4, 63), (104, "Vormittag", "Kaffee", "L", 2, 42, 9, 81),
    (105, "Abend", "Wasser", "M", 0, 30, 8, 84), (106, "Vormittag", "Tee", "M", 1, 28, 5, 60),
    (107, "Abend", "Kaffee", "XL", 0, 55, 9, 86), (108, "Vormittag", "Kaffee", "S", 3, 12, 7, 79),
    (109, "Vormittag", "Mate", "M", 1, 38, 3, 57), (110, "Abend", "Tee", "M", 0, 115, 12, 66),
    (111, "Vormittag", "Kaffee", "L", 0, 32, 10, 88), (112, "Abend", "Mate", "L", 2, 45, 1, 47),
    (113, "Vormittag", "Wasser", "S", 1, 22, 6, 72), (114, "Vormittag", "Kaffee", "M", 0, 35, 5, 62),
    (115, "Abend", "Tee", "L", 1, 48, 11, 94), (116, "Vormittag", "Tee", "M", 0, 30, 8, 76),
    (117, "Abend", "Kaffee", "M", 3, 65, 2, 52), (118, "Vormittag", "Mate", "XL", 1, 40, 6, 69),
    (119, "Abend", "Wasser", "S", 0, 52, 7, 70), (120, "Vormittag", "Kaffee", "L", 2, 25, 6, 61),
]
spalten = ["teilnehmer_id", "kurszeit", "getraenk", "tshirt", "haustiere",
           "anfahrt_min", "lernstunden", "punkte"]
umfrage = pd.DataFrame(zeilen, columns=spalten)

BLAU, ORANGE, GRAU = "#2a78d6", "#eb6834", "#b0b0b0"
ordner = "csv & excel Dateien/"

print("Mittelwert Anfahrt:", umfrage["anfahrt_min"].mean())

Mittelwert Anfahrt: 39.6


## 2 – Lücken finden

Fehlende Werte heißen in Pandas `NaN`. Wir löschen die drei längsten Anfahrtswege (52, 55, 65 Minuten), so wie am Vormittag: Wer weit fährt, hat den Fragebogen nicht abgegeben.

In [2]:
anfahrt = umfrage["anfahrt_min"].astype(float)
luecken = anfahrt.copy()
luecken[luecken.isin([52, 55, 65])] = np.nan

print("fehlende Werte:", luecken.isna().sum())
print("count():       ", luecken.count())
print("len():         ", len(luecken))

fehlende Werte: 3
count():        17
len():          20


`count()` zählt nur die vorhandenen Werte, `len()` alle Zeilen. Das ist wichtig für jede Quote: Was steht im Nenner?

> **Achtung:** `mean()` überspringt `NaN` still. Es kommt eine Zahl heraus, auch wenn Werte fehlen.

In [3]:
print("Mittelwert vollständig:", round(anfahrt.mean(), 1))
print("Mittelwert mit Lücken: ", round(luecken.mean(), 1))

Mittelwert vollständig: 39.6
Mittelwert mit Lücken:  36.5


Der Mittelwert fällt von 39,6 auf 36,5 Minuten. Die Lücken sind **systematisch**: Es fehlen genau die langen Wege. Das Ergebnis ist verzerrt, und kein Fehler wird angezeigt.

## 3 – Lücken füllen und kennzeichnen

`fillna()` füllt Lücken mit einem Wert, zum Beispiel dem Mittelwert. `dropna()` entfernt Zeilen mit Lücken.

In [4]:
gefuellt = luecken.fillna(luecken.mean())

print("Mittelwert vorher/nachher:          ", round(luecken.mean(), 1), round(gefuellt.mean(), 1))
print("Standardabweichung vorher/nachher:  ", round(luecken.std(), 1), round(gefuellt.std(), 1))
print("Zeilen nach dropna():               ", len(luecken.dropna()))

Mittelwert vorher/nachher:           36.5 36.5
Standardabweichung vorher/nachher:   22.4 20.5
Zeilen nach dropna():                17


Der Mittelwert bleibt gleich, die Streuung schrumpft von 22,4 auf 20,5. Drei erfundene Werte genau in der Mitte machen die Daten gleichförmiger, als sie sind.

Deshalb gehört zu jedem gefüllten Wert eine **Kennzeichnung**:

In [5]:
anfahrt_tabelle = pd.DataFrame({
    "anfahrt_min": gefuellt.round(1),
    "geschaetzt": luecken.isna(),      # True = Wert wurde gefüllt
})
anfahrt_tabelle[anfahrt_tabelle["geschaetzt"]]

,anfahrt_min,geschaetzt
6,36.5,True
16,36.5,True
18,36.5,True


## 4 – Dubletten

<img src="png%20Dateien/der_ausreisser.png" alt="Der Ausreißer" width="110">

**Der Ausreißer:** „Ich bin zweimal in der Tabelle. Doppelt hält besser, oder?"

In [6]:
# Person 110 wird versehentlich zweimal eingetragen
doppelt = pd.concat([umfrage, umfrage[umfrage["teilnehmer_id"] == 110]], ignore_index=True)

print("Zeilen:            ", len(doppelt))
print("Mittelwert Anfahrt:", round(doppelt["anfahrt_min"].mean(), 1))
print("Dubletten:         ", doppelt.duplicated().sum())

bereinigt = doppelt.drop_duplicates()
print("nach drop_duplicates:", len(bereinigt), "Zeilen, Mittelwert", round(bereinigt["anfahrt_min"].mean(), 1))

Zeilen:             21
Mittelwert Anfahrt: 43.2
Dubletten:          1
nach drop_duplicates: 20 Zeilen, Mittelwert 39.6


Eine einzige doppelte Zeile hebt den Mittelwert von 39,6 auf 43,2. `duplicated()` markiert jede Zeile, die schon einmal vorkam, `drop_duplicates()` entfernt sie.

Mit `subset=["teilnehmer_id"]` prüft Pandas nur bestimmte Spalten. So findest du auch Dubletten, bei denen sich ein Tippfehler in einer anderen Spalte unterscheidet.

### Übung 1

Die Übungen 1 und 2 vom Vormittag als Gegenprobe.

- **a)** Lege `s = pd.Series([2, 4, 6, 8, 10, np.nan, np.nan])` an. Berechne Mittelwert, Standardabweichung, `count()` und `len()`.
- **b)** Fülle die Lücken mit dem Mittelwert. Wie ändert sich die Standardabweichung?
- **c)** Bestellungen: `bestellung = ["B1", "B2", "B3", "B2", "B4", "B3"]`, `betrag = [120, 80, 200, 80, 100, 200]`. Berechne den Gesamtumsatz mit und ohne Dubletten.

In [7]:
# a)
s = pd.Series([2, 4, 6, 8, 10, np.nan, np.nan])
print("Mittelwert", s.mean())
print("Std.", round(s.std(), 2))
print("count", s.count(), "| len", len(s))


Mittelwert 6.0
Std. 3.16
count 5 | len 7


In [ ]:
# b)
print("Std. nach Füllen", round(s.fillna(s.mean()).std(), 2))


Std nach Füllen 2.58


In [64]:
# c)
best = pd.DataFrame({"bestellung": ["B1", "B2", "B3", "B2", "B4", "B3"],
                     "betrag": [120, 80, 200, 80, 100, 200]})
print(best)
print("Mit D.", best["betrag"].sum())
print("o D.", best.drop_duplicates()["betrag"].sum())
print(best)

  bestellung  betrag
0         B1     120
1         B2      80
2         B3     200
3         B2      80
4         B4     100
5         B3     200
Mit D. 780
o D. 500
  bestellung  betrag
0         B1     120
1         B2      80
2         B3     200
3         B2      80
4         B4     100
5         B3     200


---
# Block B – Datentypen, unplausible Werte, Schreibvarianten

## 5 – Zahlen als Text

Steht in einer Spalte auch nur ein Text, speichert Pandas die ganze Spalte als Text. Rechnen geht dann nicht. `pd.to_numeric(..., errors="coerce")` wandelt um und macht aus allem, was keine Zahl ist, `NaN`.

In [10]:
roh = pd.Series(["35", "40", "fünfzig", "28", ""])
print(roh.dtype)

zahlen = pd.to_numeric(roh, errors="coerce")
print(zahlen.tolist())
print("Mittelwert:", zahlen.mean())

str
[35.0, 40.0, nan, 28.0, nan]
Mittelwert: 34.333333333333336


„fünfzig" und das leere Feld werden zu `NaN`. Danach weißt du genau, wo du nachfassen musst.

## 6 – Unplausible Werte: der Einheitenmix

Jemand hat die Anfahrt in Stunden statt in Minuten angegeben: 0,7 statt 42.

In [11]:
einheit = umfrage["anfahrt_min"].astype(float).copy()
einheit[einheit == 42] = 0.7

print("Mittelwert:", round(einheit.mean(), 1), "| Median:", einheit.median())

# Prüfregel: Eine Anfahrt unter 1 Minute ist nicht plausibel
print(einheit[einheit < 1])

Mittelwert: 37.5 | Median: 33.5
3    0.7
Name: anfahrt_min, dtype: float64


Eine **Prüfregel** findet solche Werte zuverlässig. Weitere Regeln: Wertebereich (Alter zwischen 16 und 70), Logik (angetreten ≤ angemeldet) und die IQR-Zäune aus Tag 3.

> **Ausreißer ≠ Fehler.** Die 115 Minuten von Person 110 sind ungewöhnlich, aber echt. Die 0,7 ist ein Fehler.

## 7 – Schreibvarianten

Übung 4 vom Vormittag: Für Pandas sind „Kaffee", „kaffee" und „Kaffee " drei verschiedene Werte.

In [12]:
getraenke = pd.Series(["Kaffee", "kaffee", "Tee", "KAFFEE", "Mate", "tee", "Kaffee", "Wasser",
                       "Kaffee ", "Mate", "Tee", "kaffee"])
print("verschiedene Werte vorher:", getraenke.nunique())

sauber = getraenke.str.strip().str.title()
print("verschiedene Werte nachher:", sauber.nunique())
print(sauber.value_counts())

verschiedene Werte vorher: 8
verschiedene Werte nachher: 4
Kaffee    6
Tee       3
Mate      2
Wasser    1
Name: count, dtype: int64


- `.str.strip()` entfernt Leerzeichen am Anfang und Ende.
- `.str.title()` schreibt jedes Wort groß. Alternativen: `.str.lower()`, `.str.upper()`.
- `.str.replace("alt", "neu")` ersetzt Zeichen, zum Beispiel ein Euro-Zeichen durch nichts.

## 8 – Zerlegen, suchen, Datum

Drei weitere Werkzeuge, die du in echten Daten oft brauchst:

In [13]:
namen = pd.Series(["Mia Schulz", "Jonas Weber", "Lena Koch"])
print(namen.str.split(" ", expand=True))          # zerlegen in Spalten
print()
print(namen[namen.str.contains("Weber")])          # filtern nach Textteil
print()
daten = pd.to_datetime(pd.Series(["2026-09-01", "2026-10-15"]))
print(daten.dt.month.tolist())                      # Text wird zum Datum, Monat auslesen

       0       1
0    Mia  Schulz
1  Jonas   Weber
2   Lena    Koch

1    Jonas Weber
dtype: str

[9, 10]


### Übung 2

Die Anmeldeliste der Datenwald-Akademie ist aus einem Formular kopiert. Führe zuerst die Zelle aus.

- **a)** Vereinheitliche die Spalte `kurs` und zähle die Anmeldungen pro Kurs.
- **b)** Wandle `gebuehr` in Zahlen um. Entferne vorher das Euro-Zeichen. Wie hoch ist die mittlere Gebühr? Wie viele Werte fehlen?
- **c)** Entferne die Leerzeichen in `name` und lege eine Spalte `vorname` an.

In [14]:
anmeldungen = pd.DataFrame({
    "name": [" Mia Schulz", "Jonas Weber ", "Lena Koch", "Tim Bauer", "Sara Yilmaz", "Paul Richter"],
    "kurs": ["Python", "python ", "SQL", "PYTHON", "sql", "Excel"],
    "gebuehr": ["450", "450", "380 €", "450", "keine Angabe", "290"],
})
anmeldungen

,name,kurs,gebuehr
0,Mia Schulz,Python,450
1,Jonas Weber,python,450
2,Lena Koch,SQL,380 €
3,Tim Bauer,PYTHON,450
4,Sara Yilmaz,sql,keine Angabe
5,Paul Richter,Excel,290


In [ ]:
# a)
anmeldungen["kurs"] = anmeldungen["kurs"].str.strip().str.title()
print(anmeldungen["kurs"].value_counts())
# ggf. .str.upper() für SQL

kurs
Python    3
Sql       2
Excel     1
Name: count, dtype: int64


In [76]:
# b)
# anmeldungen["gebuehr"] = pd.to_numeric(anmeldungen["gebuehr"].str.replace("€", "").str.strip(), errors="coerce")
print(anmeldungen["gebuehr"].tolist())
print("mittlere Gebühr", anmeldungen["gebuehr"].mean())
print("fehlend", anmeldungen["gebuehr"].isna().sum())


[450.0, 450.0, 380.0, 450.0, nan, 290.0]
mittlere Gebühr 404.0
fehlend 1


In [78]:
# c)
anmeldungen["name"] = anmeldungen["name"].str.strip()
anmeldungen["vorname"] = anmeldungen["name"].str.split(" ").str[0]
anmeldungen

,name,kurs,gebuehr,vorname
0,Mia Schulz,Python,450.0,Mia
1,Jonas Weber,Python,450.0,Jonas
2,Lena Koch,Sql,380.0,Lena
3,Tim Bauer,Python,450.0,Tim
4,Sara Yilmaz,Sql,NaN,Sara
5,Paul Richter,Excel,290.0,Paul


---
# Block C – Tabellen zusammenführen mit `merge`

## 9 – Zwei Tabellen, ein Schlüssel

In der Praxis liegen Daten selten in einer Tabelle. Die TechAcademy GmbH hat Teilnehmende, Buchungen und Kurse getrennt gespeichert, verbunden über **Schlüsselspalten** wie in einer Datenbank.

In [18]:
teilnehmer = pd.DataFrame({
    "TeilnehmerID": [101, 102, 103, 104, 105],
    "Name": ["Mia Schulz", "Jonas Weber", "Lena Koch", "Tim Bauer", "Sara Yilmaz"],
    "Wohnort": ["Berlin", "Hamburg", "München", "Berlin", "Frankfurt"],
})
buchungen = pd.DataFrame({
    "BuchungsID": [1001, 1002, 1003, 1004, 1005, 1006],
    "TeilnehmerID": [101, 102, 102, 103, 104, 999],     # 999 gibt es nicht
    "KursID": ["K01", "K02", "K01", "K03", "K01", "K02"],
})
kurse = pd.DataFrame({
    "KursID": ["K01", "K02", "K03"],
    "Kursname": ["Python Grundlagen", "SQL für Einsteiger", "Data Analysis"],
    "Trainer": ["Sandra Meier", "Tobias Lange", "Michael Brandt"],
})

inner = pd.merge(buchungen, teilnehmer, on="TeilnehmerID", how="inner")
print(inner)
print("Zeilen:", len(inner))

   BuchungsID  TeilnehmerID KursID         Name  Wohnort
0        1001           101    K01   Mia Schulz   Berlin
1        1002           102    K02  Jonas Weber  Hamburg
2        1003           102    K01  Jonas Weber  Hamburg
3        1004           103    K03    Lena Koch  München
4        1005           104    K01    Tim Bauer   Berlin
Zeilen: 5


`how="inner"` (der Standard) behält nur Zeilen, deren Schlüssel in **beiden** Tabellen vorkommt. Buchung 1006 mit Teilnehmer 999 ist verschwunden, ohne Warnung.

## 10 – `left` und die Kontrolle mit `indicator=True`

`how="left"` behält alle Zeilen der linken Tabelle. Fehlt rechts ein Partner, steht `NaN`. Mit `indicator=True` zeigt die Spalte `_merge`, woher jede Zeile kommt.

In [19]:
links = pd.merge(buchungen, teilnehmer, on="TeilnehmerID", how="left", indicator=True)
print(links)
print()
print(links["_merge"].value_counts())

   BuchungsID  TeilnehmerID KursID         Name  Wohnort     _merge
0        1001           101    K01   Mia Schulz   Berlin       both
1        1002           102    K02  Jonas Weber  Hamburg       both
2        1003           102    K01  Jonas Weber  Hamburg       both
3        1004           103    K03    Lena Koch  München       both
4        1005           104    K01    Tim Bauer   Berlin       both
5        1006           999    K02          NaN      NaN  left_only

_merge
both          5
left_only     1
right_only    0
Name: count, dtype: int64


`left_only` heißt: Diese Buchung hat keinen Teilnehmer. Vor jeder Auswertung lohnt dieser Blick.

Danach kannst du weitermergen, zum Beispiel die Kurse dazu:

In [20]:
master = pd.merge(inner, kurse, on="KursID", how="left")
master[["Name", "Kursname", "Trainer"]]

,Name,Kursname,Trainer
0,Mia Schulz,Python Grundlagen,Sandra Meier
1,Jonas Weber,SQL für Einsteiger,Tobias Lange
2,Jonas Weber,Python Grundlagen,Sandra Meier
3,Lena Koch,Data Analysis,Michael Brandt
4,Tim Bauer,Python Grundlagen,Sandra Meier


## 11 – `outer`, `right`, `left_on`, `suffixes`

In [21]:
# outer: alle Zeilen aus beiden Tabellen (hier: Buchung 999 UND Sara ohne Buchung)
print(len(pd.merge(buchungen, teilnehmer, on="TeilnehmerID", how="outer")), "Zeilen bei outer")
# right: alle Zeilen der rechten Tabelle (wie left mit vertauschten Tabellen)

# left_on / right_on: der Schlüssel heißt in beiden Tabellen anders
buchungen_v2 = buchungen.rename(columns={"TeilnehmerID": "TN"})
print(pd.merge(buchungen_v2, teilnehmer, left_on="TN", right_on="TeilnehmerID").shape)

# suffixes: gleiche Spaltennamen außer dem Schlüssel bekommen Endungen
a = pd.DataFrame({"id": [1, 2], "datum": ["2026-01-10", "2026-02-01"]})
b = pd.DataFrame({"id": [1, 2], "datum": ["2026-03-01", "2026-03-15"]})
print(pd.merge(a, b, on="id", suffixes=("_anmeldung", "_pruefung")))

7 Zeilen bei outer
(5, 6)
   id datum_anmeldung datum_pruefung
0   1      2026-01-10     2026-03-01
1   2      2026-02-01     2026-03-15


## 12 – Doppelte Schlüssel vermehren Zeilen

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="120">

> **Herr Bauchgefühl:** „Nach dem Merge haben wir mehr Buchungen als vorher. Das Geschäft läuft!"

In der Kurstabelle steht K01 versehentlich zweimal.

In [22]:
kurse_doppelt = pd.concat([kurse, kurse.iloc[[0]]], ignore_index=True)
print(kurse_doppelt)
print()

ergebnis = pd.merge(buchungen, kurse_doppelt, on="KursID")
print("Buchungen vorher:", len(buchungen), "| nach dem Merge:", len(ergebnis))

  KursID            Kursname         Trainer
0    K01   Python Grundlagen    Sandra Meier
1    K02  SQL für Einsteiger    Tobias Lange
2    K03       Data Analysis  Michael Brandt
3    K01   Python Grundlagen    Sandra Meier

Buchungen vorher: 6 | nach dem Merge: 9


Jede Buchung für K01 trifft zweimal, aus 6 werden 9 Zeilen. Summen und Anzahlen danach sind falsch, und Pandas warnt nicht.

Schutz: Vorher `duplicated()` auf dem Schlüssel prüfen, oder `validate="many_to_one"` mitgeben. Dann bricht der Merge mit einer Fehlermeldung ab, statt still falsche Zeilen zu erzeugen.

### Übung 3

Führe zuerst die Zelle aus. In `staedte` heißt der Schlüssel `StadtNr`, in `personen` heißt er `StadtID`.

- **a)** Verbinde `personen` und `staedte` mit einem inner join. Wer fehlt und warum?
- **b)** Verbinde mit einem left join und `indicator=True`. Wie viele Zeilen sind `left_only`?
- **c)** Prüfe vor dem Merge, ob ein Schlüssel in `staedte` doppelt vorkommt.

In [23]:
personen = pd.DataFrame({
    "TeilnehmerID": [301, 302, 303, 304, 305],
    "Name": ["Finn Müller", "Lea Wagner", "Omar Hassan", "Jana Becker", "Timo Schulz"],
    "StadtID": [10, 20, 10, 30, 20],
})
staedte = pd.DataFrame({
    "StadtNr": [10, 20, 40],
    "Stadt": ["Berlin", "Hamburg", "München"],
    "Bundesland": ["Berlin", "Hamburg", "Bayern"],
})

In [79]:
# a)
inner_staedte = pd.merge(personen, staedte, left_on="StadtID", right_on="StadtNr", how="inner")
print(inner_staedte)


   TeilnehmerID         Name  StadtID  StadtNr    Stadt Bundesland
0           301  Finn Müller       10       10   Berlin     Berlin
1           302   Lea Wagner       20       20  Hamburg    Hamburg
2           303  Omar Hassan       10       10   Berlin     Berlin
3           305  Timo Schulz       20       20  Hamburg    Hamburg


In [80]:
# b)
links_staedte = pd.merge(personen, staedte, left_on="StadtID", right_on="StadtNr", how="left", indicator=True)
print(links_staedte)
print(links_staedte["_merge"].value_counts())

   TeilnehmerID         Name  StadtID  StadtNr    Stadt Bundesland     _merge
0           301  Finn Müller       10     10.0   Berlin     Berlin       both
1           302   Lea Wagner       20     20.0  Hamburg    Hamburg       both
2           303  Omar Hassan       10     10.0   Berlin     Berlin       both
3           304  Jana Becker       30      NaN      NaN        NaN  left_only
4           305  Timo Schulz       20     20.0  Hamburg    Hamburg       both
_merge
both          4
left_only     1
right_only    0
Name: count, dtype: int64


In [83]:
# c)
print("doppelte Schlüssel", staedte["StadtNr"].duplicated().sum())


doppelte Schlüssel 0


---
# Block D – Die Pivot-Tabelle

## 13 – `pivot_table` mit vier Zutaten

| Zutat | Pandas | Power BI (Matrix) | Excel (PivotTable) |
|---|---|---|---|
| Zeilen | `index` | Zeilen | Zeilen |
| Spalten | `columns` | Spalten | Spalten |
| Werte | `values` | Werte | Werte |
| Zusammenfassung | `aggfunc` | Measure / Zusammenfassung | Wertfeldeinstellungen |

Das Beispiel vom Vormittag: mittlere Punkte nach Kurszeit und Getränk.

In [27]:
pivot = umfrage.pivot_table(index="kurszeit", columns="getraenk", values="punkte",
                            aggfunc="mean", margins=True, margins_name="gesamt")
pivot.round(1)

getraenk,Kaffee,Mate,Tee,Wasser,gesamt
kurszeit,,,,,
Abend,69.0,47.0,72.7,77.0,69.6
Vormittag,70.2,63.0,68.0,72.0,68.2
gesamt,69.9,59.0,70.8,75.3,68.8


`margins=True` fügt die Zeile und Spalte „gesamt" hinzu. Die Randwerte sind **gewichtet**: 68,8 ist der Mittelwert aller 20 Personen, nicht der Mittelwert der Zellen.

<img src="png%20Dateien/der_ausreisser.png" alt="Der Ausreißer" width="110">

**Der Ausreißer:** „Eine Zelle, eine Person, und alle glauben an einen Trend."

Deshalb gehört die Anzahl immer dazu:

In [28]:
anzahl = umfrage.pivot_table(index="kurszeit", columns="getraenk", values="punkte",
                             aggfunc="count", margins=True, margins_name="gesamt")
anzahl

getraenk,Kaffee,Mate,Tee,Wasser,gesamt
kurszeit,,,,,
Abend,2,1,3,2,8
Vormittag,6,3,2,1,12
gesamt,8,4,5,3,20


Mate am Abend und Wasser am Vormittag: jeweils **eine** Person. Die 47,0 Punkte bei Mate/Abend sind die Punktzahl von Person 112, kein Trend.

Beides auf einmal geht mit einer Liste:

In [29]:
umfrage.pivot_table(index="kurszeit", columns="getraenk", values="punkte",
                    aggfunc=["mean", "count"]).round(1)

mean                     count                
getraenk  Kaffee  Mate   Tee Wasser Kaffee Mate Tee Wasser
kurszeit                                                  
Abend       69.0  47.0  72.7   77.0      2    1   3      2
Vormittag   70.2  63.0  68.0   72.0      6    3   2      1

## 14 – Pivot, `crosstab` und `groupby`

- Die Kreuztabelle aus Tag 1 ist eine Pivot-Tabelle, die nur zählt.
- `pivot_table` rechnet **standardmäßig den Mittelwert**. Für Summen `aggfunc="sum"`.
- Dasselbe wie die Pivot-Tabelle geht mit `groupby` und `unstack()`.

In [ ]:
print(pd.crosstab(umfrage["kurszeit"], umfrage["getraenk"]))
print()
print(umfrage.groupby(["kurszeit", "getraenk"])["punkte"].mean().unstack().round(1))   # stack() -> Spalten zu Liste = "stapeln in die Länge"
                                                                                       # unstack() -> "auspacken in die Breite"

getraenk   Kaffee  Mate  Tee  Wasser
kurszeit                            
Abend           2     1    3       2
Vormittag       6     3    2       1

getraenk   Kaffee  Mate   Tee  Wasser
kurszeit                             
Abend        69.0  47.0  72.7    77.0
Vormittag    70.2  63.0  68.0    72.0


### Übung 4

Die Prüfungsdaten der TechAcademy. Führe zuerst die Zelle aus.

- **a)** Erstelle eine Pivot-Tabelle: mittlere Note nach Kurs (Zeilen) und Kursformat (Spalten), mit Randwerten.
- **b)** Erstelle dieselbe Tabelle mit der Anzahl. Welche Zellen sind zu klein für eine Aussage?
- **c)** Erstelle eine Pivot-Tabelle mit der Anzahl bestandener Prüfungen nach Trainer und Monat. Leere Felder sollen 0 zeigen (`fill_value=0`).

In [31]:
pruefungen = pd.DataFrame({
    "Kurs": ["Python Grundlagen", "SQL für Einsteiger", "Python Grundlagen", "Data Analysis", "Linux Basics",
             "SQL für Einsteiger", "Data Analysis", "Python Grundlagen", "Linux Basics", "Data Analysis",
             "Data Analysis", "Python Grundlagen", "SQL für Einsteiger", "Linux Basics", "Linux Basics"],
    "Kursformat": ["Präsenz", "Online", "Präsenz", "Online", "Präsenz", "Online", "Präsenz", "Online",
                   "Präsenz", "Online", "Online", "Präsenz", "Präsenz", "Online", "Online"],
    "Trainer": ["Sandra Meier", "Tobias Lange", "Sandra Meier", "Michael Brandt", "Yara Özcan",
                "Tobias Lange", "Michael Brandt", "Sandra Meier", "Yara Özcan", "Michael Brandt",
                "Michael Brandt", "Sandra Meier", "Tobias Lange", "Yara Özcan", "Yara Özcan"],
    "Monat": ["Januar", "Januar", "Januar", "Februar", "Februar", "Februar", "März", "März", "März", "März",
              "April", "April", "April", "April", "April"],
    "Note": [2.0, 1.7, 3.3, 1.0, 2.7, 2.3, 1.3, 4.0, 2.0, 1.7, 1.3, 2.7, 2.0, 3.7, 4.3],
    "Bestanden": [True, True, True, True, True, True, True, False, True, True, True, True, True, False, False],
})
pruefungen.head()

,Kurs,Kursformat,Trainer,Monat,Note,Bestanden
0,Python Grundlagen,Präsenz,Sandra Meier,Januar,2.0,True
1,SQL für Einsteiger,Online,Tobias Lange,Januar,1.7,True
2,Python Grundlagen,Präsenz,Sandra Meier,Januar,3.3,True
3,Data Analysis,Online,Michael Brandt,Februar,1.0,True
4,Linux Basics,Präsenz,Yara Özcan,Februar,2.7,True


In [85]:
# a)
pruefungen.pivot_table(index="Kurs", columns="Kursformat", values="Note", margins=True, margins_name="gesamt").round(2)


Kursformat,Online,Präsenz,gesamt
Kurs,,,
Data Analysis,1.33,1.30,1.32
Linux Basics,4.00,2.35,3.18
Python Grundlagen,4.00,2.67,3.00
SQL für Einsteiger,2.00,2.00,2.00
gesamt,2.50,2.29,2.40


In [87]:
# b)
print(pruefungen.pivot_table(index="Kurs", columns="Kursformat", values="Note", aggfunc="count"))

Kursformat          Online  Präsenz
Kurs                               
Data Analysis            3        1
Linux Basics             2        2
Python Grundlagen        1        3
SQL für Einsteiger       2        1


In [88]:
# c)
reihenfolge = ["Januar", "Februar", "März", "April"]
bestanden = pruefungen.pivot_table(index="Trainer", columns="Monat", values="Bestanden", aggfunc="sum", fill_value=0)
bestanden[reihenfolge]


Monat,Januar,Februar,März,April
Trainer,,,,
Michael Brandt,0,1,2,1
Sandra Meier,2,0,0,1
Tobias Lange,1,1,0,1
Yara Özcan,0,1,1,0


---
# Block E – Das Simpson-Paradoxon

## 15 – Zwei Lernplattformen

> **Herr Bauchgefühl:** „Plattform B ist besser, 75 % gegen 67,5 %!"

Die Daten kommen als zwei Tabellen: `teilnehmende` (wer nutzt welche Plattform, mit welchem Vorwissen) und `ergebnisse` (wer hat bestanden). Die Personen-IDs sind erfunden, die Zahlen ergeben genau die Werte vom Vormittag.

In [35]:
def gruppe(start, n, plattform, vorwissen, bestanden):
    ids = range(start, start + n)
    pers = pd.DataFrame({"person_id": ids, "plattform": plattform, "vorwissen": vorwissen})
    erg = pd.DataFrame({"person_id": ids, "bestanden": [1] * bestanden + [0] * (n - bestanden)})
    return pers, erg

teile = [gruppe(1, 90, "A", "Anfänger", 54), gruppe(91, 30, "A", "Fortgeschrittene", 27),
         gruppe(121, 30, "B", "Anfänger", 15), gruppe(151, 90, "B", "Fortgeschrittene", 75)]
teilnehmende = pd.concat([p for p, _ in teile], ignore_index=True)
ergebnisse = pd.concat([e for _, e in teile], ignore_index=True)

# Ergebnisliste mischen, wie sie aus einem anderen System käme
ergebnisse = ergebnisse.sample(frac=1, random_state=1).reset_index(drop=True)

print(teilnehmende.head(3))
print(ergebnisse.head(3))

   person_id plattform vorwissen
0          1         A  Anfänger
1          2         A  Anfänger
2          3         A  Anfänger
   person_id  bestanden
0        229          0
1        195          1
2         89          0


Erst mergen, dann auswerten. `validate="one_to_one"` sichert ab, dass jede Person genau einmal vorkommt.

In [36]:
plattformen = pd.merge(teilnehmende, ergebnisse, on="person_id", how="left", validate="one_to_one")
print("Zeilen:", len(plattformen))

# Eine Dimension: Plattform
(plattformen.pivot_table(index="plattform", values="bestanden", aggfunc="mean") * 100).round(1)

Zeilen: 240


,bestanden
plattform,
A,67.5
B,75.0


Mit einer Dimension scheint B besser. Jetzt kommt das Vorwissen dazu:

In [37]:
(plattformen.pivot_table(index="vorwissen", columns="plattform", values="bestanden",
                         aggfunc="mean", margins=True, margins_name="gesamt") * 100).round(1)

plattform,A,B,gesamt
vorwissen,,,
Anfänger,60.0,50.0,57.5
Fortgeschrittene,90.0,83.3,85.0
gesamt,67.5,75.0,71.2


<img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats Maulwurf" width="140">

**Mats Maulwurf:** „Ich bin das Vorwissen. Ohne mich sieht B besser aus."

A ist bei den Anfängern besser (60 gegen 50 %) **und** bei den Fortgeschrittenen (90 gegen 83,3 %). Nur insgesamt liegt B vorn. Die Erklärung steckt in der Anzahl:

In [38]:
pd.crosstab(plattformen["vorwissen"], plattformen["plattform"], margins=True)

plattform,A,B,All
vorwissen,,,
Anfänger,90,30,120
Fortgeschrittene,30,90,120
All,120,120,240


A hat 90 Anfänger, B nur 30. Anfänger bestehen seltener, also drückt das die Gesamtquote von A. Die Randspalte „gesamt" ist gewichtet, genau wie in Abschnitt 13.

> **Merksatz:** Eine Dimension mehr in der Pivot-Tabelle kann die Aussage umdrehen.

### Übung 5

Arbeite mit `plattformen`.

- **a)** Berechne die Gesamtquote von A von Hand aus den Gruppen: (54 + 27) / 120. Stimmt sie mit der Randspalte überein?
- **b)** Berechne den **ungewichteten** Mittelwert der beiden Gruppenquoten von A. Warum ist er falsch?
- **c)** Formuliere in einem Kommentar eine ehrliche Aussage für die Geschäftsführung.

In [91]:
# a)
print("per Hand", (54 + 27)/ 120 * 100)
print("PD", plattformen[plattformen["plattform"] == "A"]["bestanden"].mean() * 100)

per Hand 67.5
PD 67.5


In [92]:
# b)
print("ungewichtet", (60 + 90)/ 2)

ungewichtet 75.0


In [41]:
# c)


---
# Block F – Aufwärmübungen und Projekt

- **Übung 6 bis 8:** kurze Aufwärmübungen zum Vormittag (KPI, Pareto, ABC)
- **Übung 9:** das Projekt „Akademie-Laden" mit One-Pager
- **Zusatz:** weitere Merge-Übungen aus früheren Kursen

## 16 – Aufwärmübungen

### Übung 6 – KPI-Steckbrief und Prozent-Fallen (zum Vormittag)

- **a)** Fülle den Steckbrief unten aus (Doppelklick auf die Zelle) und berechne die Bestehensquote mit beiden Nennern: 120 angemeldet, 110 angetreten, 88 bestanden.
- **b)** Eine Quote steigt von 60 % auf 75 %. Wie viele Prozentpunkte, wie viel Prozent?
- **c)** Ein Umsatz steigt um 50 % und fällt dann um 50 %. Wie viel Prozent sind es insgesamt?
- **d)** Die Akademie wächst in 7 Jahren von 52 auf 68 Teilnehmende. Wie hoch ist das Wachstum pro Jahr? *Hinweis: geometrisches Mittel, `(68 / 52) ** (1 / 7) - 1`.*

| Feld | Eintrag |
|---|---|
| Name | |
| Zweck | |
| Formel (Zähler / Nenner) | |
| Datenquelle | |
| Zeitraum | |
| Einheit | |
| Zielwert | |
| Verantwortlich | |

In [42]:
# a)


In [43]:
# b)


In [44]:
# c)


In [45]:
# d)


### Übung 7 – Das Pareto-Diagramm (zum Vormittag)

Die Support-Tickets aus dem Skript: Passwort 46, VPN 28, Drucker 12, E-Mail 6, Software 4, Hardware 2, Sonstiges 2.

- **a)** Lege die Tickets als Series an, sortiere absteigend und berechne den kumulierten Anteil in Prozent (`cumsum()`).
- **b)** Zeichne das Pareto-Diagramm: Balken für die Anzahl, eine Linie für den kumulierten Anteil auf einer zweiten Achse (`ax.twinx()`), eine gestrichelte Linie bei 80 %.
- **c)** In Notebook 5 war die zweite y-Achse ein Trick. Warum ist sie hier ehrlich?

In [46]:
# a)


In [47]:
# b)


In [48]:
# c)


### Übung 8 – ABC-Analyse (zum Vormittag)

Der Merch-Shop der Akademie aus dem Skript. Führe zuerst die Zelle aus.

- **a)** Sortiere nach Umsatz, berechne Anteil und kumulierten Anteil in Prozent.
- **b)** Teile die Produkte mit `pd.cut` in A (bis 80 %), B (bis 95 %) und C ein. *Hinweis: `bins=[0, 80, 95, np.inf]`.*
- **c)** Wie viele Produkte sind A, und welchen Anteil am Umsatz bringen sie?

In [49]:
shop = pd.DataFrame({
    "produkt": ["Rucksack", "Headset", "USB-Hub", "Notizbuch", "Becher", "Stifte", "Sticker", "Lanyard"],
    "umsatz": [4200, 2600, 1100, 700, 500, 400, 300, 200],
})

In [50]:
# a)


In [51]:
# b)


In [52]:
# c)


## 17 – Projekt: Der Akademie-Laden

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="120">

> **Herr Bauchgefühl:** „Der Laden? Läuft doch. Ich schicke der Geschäftsführung einfach die Rohdaten, die Zahlen sprechen für sich."

Die Akademie verkauft im Pausenladen Obst, Gemüse und Milchprodukte. Die Geschäftsführung möchte wissen: **Welche Produkte tragen den Umsatz, und worauf sollen wir uns konzentrieren?**

Du bekommst zwei Dateien aus zwei verschiedenen Systemen, beide im Ordner `csv & excel Dateien`:

- `produkte.csv`: Stammdaten (ID, Name, Preis, Kategorie)
- `verkaufsdaten.csv`: Verkäufe eines Monats (Artikel-ID, verkaufte Einheiten, Umsatz)

Die Daten sind nicht sauber. Diesmal gibt es keine Schritt-für-Schritt-Hinweise. Du kennst alle Werkzeuge aus diesem Notebook.

**Regeln für das Projekt**

- Führe ein **Bereinigungsprotokoll**: Was hast du geändert, warum, wie viele Zeilen betrifft es?
- Lösche nichts stillschweigend. Jede entfernte oder geänderte Zeile kommt ins Protokoll.
- Wenn du eine Entscheidung triffst (zum Beispiel einen Wert ausschließt), begründe sie.

### Übung 9 – Akademie-Laden

- **Teil 1 – Erster Blick:** Lies beide Dateien ein. Prüfe Zeilen, Datentypen, Lücken und Dubletten.
- **Teil 2 – Schlüssel:** Die IDs sind uneinheitlich geschrieben. Bringe sie in beiden Tabellen in die Form `P-1001`.
- **Teil 3 – Dubletten:** Entferne doppelte Stammdaten. Achtung: Kommt dieselbe ID in den Verkaufsdaten zweimal vor, ist das vielleicht kein Fehler. Entscheide und begründe.
- **Teil 4 – Plausibilität:** Stelle Prüfregeln auf. Hinweise: Umsatz = Einheiten × Preis; Einheiten und Preise können nicht negativ sein. Wie gehst du mit einer Lücke um, die sich aus den anderen Spalten berechnen lässt?
- **Teil 5 – Merge:** Verbinde Verkäufe und Produkte. Prüfe mit `indicator=True`, ob jeder Verkauf ein Produkt gefunden hat.
- **Teil 6 – Kennzahl:** Definiere eine KPI mit Steckbrief (als Markdown) und berechne sie. Vorschlag: Umsatzanteil je Kategorie. Stelle das Ergebnis als Pivot-Tabelle mit Umsatz **und** Anzahl der Produkte dar.
- **Teil 7 – ABC und Diagramm:** Führe eine ABC-Analyse nach Produktname durch. Zeichne ein Pareto-Diagramm: Die A-Produkte farbig, den Rest grau, die Kernaussage als Titel. Speichere es mit `fig.savefig("akademie_laden.png", dpi=150)`.
- **Teil 8 – One-Pager:** Schreibe den Bericht in die Markdown-Zelle am Ende.

In [53]:
# Teil 1


In [54]:
# Teil 2


In [55]:
# Teil 3


In [56]:
# Teil 4


In [57]:
# Teil 5


**KPI-Steckbrief** (Doppelklick und ausfüllen)

| Feld | Eintrag |
|---|---|
| Name | |
| Zweck | |
| Formel (Zähler / Nenner) | |
| Datenquelle | |
| Zeitraum | |
| Einheit | |
| Zielwert | |
| Verantwortlich | |

In [58]:
# Teil 6


In [59]:
# Teil 7


**One-Pager für die Geschäftsführung** (Doppelklick und ausfüllen)

#### Kernaussage (ein Satz mit Zahl)

…

<img src="akademie_laden.png" width="600">

#### Belege (2–3 Sätze)

…

#### Grenzen der Aussage

…

#### Empfehlung

…

<img src="png%20Dateien/etta_elster.png" alt="Etta" width="110"> <img src="png%20Dateien/nils_eichhoernchen.png" alt="Nils" width="110">

**Etta und Nils:** „Erst die Frage, dann die Zahl, dann das Bild."

**Checkliste für deinen One-Pager**

- [ ] Die Überschrift ist ein ganzer Satz mit Zahl.
- [ ] Genau ein Diagramm, die Aussage steht im Titel, das Wichtige ist farbig.
- [ ] Das passende Lagemaß, mit Streuung oder Anzahl, wo nötig.
- [ ] Keine Diagramm-Tricks (Achse, Fläche, Ausschnitt).
- [ ] Keine Ursache behauptet, wo nur ein Zusammenhang gemessen wurde.
- [ ] Datenbasis und Bereinigung stehen in den Grenzen.

## 18 – Zusatz: weitere Merge-Übungen

Wer schneller fertig ist: Die Übungen 5, 6 und 8 aus dem früheren Pandas-Kurs (Merge mit Städten, Kurse und Bewertungen, Merge mit Top-N) liegen als eigene Notebooks bei deiner Dozentin.

---
## 19 – Zusammenfassung

| Aufgabe | Befehl |
|---|---|
| Lücken zählen | `df.isna().sum()`, `count()` gegen `len()` |
| Lücken füllen / entfernen | `fillna(wert)`, `dropna()`; gefüllte Werte kennzeichnen |
| Dubletten | `duplicated()`, `drop_duplicates()`, `subset=[...]` |
| Text in Zahl | `pd.to_numeric(spalte, errors="coerce")` |
| Text bereinigen | `.str.strip()`, `.str.title()`, `.str.upper()`, `.str.replace()` |
| Text zerlegen / suchen | `.str.split()`, `.str.contains()` |
| Datum | `pd.to_datetime()`, `.dt.month` |
| Merge | `pd.merge(a, b, on="schluessel", how="inner" / "left")` |
| Merge prüfen | `indicator=True`, `validate="one_to_one"` |
| Andere Schlüsselnamen | `left_on=...`, `right_on=...` |
| Pivot-Tabelle | `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| Randwerte / Anzahl | `margins=True`, `aggfunc=["mean", "count"]`, `fill_value=0` |
| Kumulieren / ABC | `cumsum()`, `pd.cut(..., bins=[0, 80, 95, np.inf])` |

**Checkliste für Kennzahlen:** Ist der Nenner klar? Ist die Basis groß genug? Prozent oder Prozentpunkte? Richtig gewichtet? Fehlt eine Dimension?

---
*Material erstellt von Julia Brutskaya*